### Import Libraries

In [1]:
import numpy as np
import pandas as pd
import re
from ydata_profiling import ProfileReport

C:\Users\speedlink\AppData\Local\Temp\ipykernel_14612\3089340042.py:4: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [2]:
category_data=pd.read_csv("F:\A7med els3eed\Digitara - Data Analysis\Health Retail Stores\After Power Query\Categories.csv")
sales_data=pd.read_csv("F:\A7med els3eed\Digitara - Data Analysis\Health Retail Stores\After Power Query\AllSales.csv")
customer_data=pd.read_csv("F:\A7med els3eed\Digitara - Data Analysis\Health Retail Stores\After Power Query\CustomerMaster.csv")
employees_data=pd.read_csv("F:\A7med els3eed\Digitara - Data Analysis\Health Retail Stores\After Power Query\Employees.csv")
products_data=pd.read_csv("F:\A7med els3eed\Digitara - Data Analysis\Health Retail Stores\After Power Query\ProductCatalog.csv")
stock_data=pd.read_csv("F:\A7med els3eed\Digitara - Data Analysis\Health Retail Stores\After Power Query\Stock.csv")
prescriptions_data=pd.read_csv("F:\A7med els3eed\Digitara - Data Analysis\Health Retail Stores\After Power Query\Prescriptions.csv")
stores_data=pd.read_csv("F:\A7med els3eed\Digitara - Data Analysis\Health Retail Stores\After Power Query\Stores.csv")
suppliers_data=pd.read_csv("F:\A7med els3eed\Digitara - Data Analysis\Health Retail Stores\After Power Query\Suppliers.csv")

##### 1) All Sales Table

In [3]:
sales_data.head(10)

,Sale Id,Store Id,Employee Id,Product Id,Customer Id,Quantity,Sale Date,Unit Price,Discount Pct,Payment Method,Notes
0,187,2,32,1379,270,1,9/30/2025,148.36,4.7,Credit Card,NaN
1,308,2,89,1683,176,1,10/31/2024,117.27,9.6,Credit Card,NaN
2,482,2,23,2530,120,1,3/16/2025,270.38,9.6,Credit Card,NaN
3,509,2,28,3314,44,1,11/10/2025,176.67,7.5,Credit Card,NaN
4,527,2,28,3654,190,1,6/15/2025,33.13,3.8,Credit Card,NaN
5,545,2,43,968,64,1,2/23/2025,230.71,8.4,Credit Card,NaN
6,669,2,103,2572,1124,1,11/18/2024,14.54,10.0,Credit Card,NaN
7,749,2,103,4201,110,1,12/3/2024,155.80,9.8,Credit Card,NaN
8,976,2,23,169,2,1,11/19/2024,41.76,7.7,Credit Card,NaN
9,978,2,103,4917,1324,1,1/27/2025,135.63,3.3,Credit Card,NaN


In [4]:
sales_data.shape

(9900, 11)

In [5]:
sales_data['Notes'].unique()

array([nan, 'Promo code LOYALTY20 applied at checkout',
       'Price matched to competitor flyer',
       'Promo code SEASON25 applied at checkout',
       'Promo code SAVE10 applied at checkout',
       'Loyalty tier discount applied',
       'Promo code WELCOME15 applied at checkout'], dtype=object)

In [ ]:
profile=ProfileReport(sales_data,title="Sales Data Report",explorative=True)
profile.to_file("sales_data_report.html")
profile

In [7]:
def extract_promo(note):
    if pd.isna(note) or note=="":
        return pd.Series({"Promo Code":None, "Discount Reason":None})
    promo_code_match=re.search(r'Promo code (\w+) applied at checkout',note)
    if promo_code_match:
        return pd.Series({"Promo Code":promo_code_match.group(1), "Discount Reason":'Promo code'})
    if 'competitor' in note.lower():
        return pd.Series({"Promo Code":None, "Discount Reason":'Competitor Price Match'})
    if 'Loyalty' in note:
        return pd.Series({"Promo Code":None, "Discount Reason":'Loyalty Discount'})
    return pd.Series({"Promo Code":None, "Discount Reason":'other'})

extracted=sales_data['Notes'].apply(extract_promo)
sales_data=pd.concat([sales_data,extracted],axis=1)    

In [8]:
sales_data.drop(columns=['Notes'],inplace=True)

In [9]:
sales_data.head(10)

,Sale Id,Store Id,Employee Id,Product Id,Customer Id,Quantity,Sale Date,Unit Price,Discount Pct,Payment Method,Promo Code,Discount Reason
0,187,2,32,1379,270,1,9/30/2025,148.36,4.7,Credit Card,None,None
1,308,2,89,1683,176,1,10/31/2024,117.27,9.6,Credit Card,None,None
2,482,2,23,2530,120,1,3/16/2025,270.38,9.6,Credit Card,None,None
3,509,2,28,3314,44,1,11/10/2025,176.67,7.5,Credit Card,None,None
4,527,2,28,3654,190,1,6/15/2025,33.13,3.8,Credit Card,None,None
5,545,2,43,968,64,1,2/23/2025,230.71,8.4,Credit Card,None,None
6,669,2,103,2572,1124,1,11/18/2024,14.54,10.0,Credit Card,None,None
7,749,2,103,4201,110,1,12/3/2024,155.80,9.8,Credit Card,None,None
8,976,2,23,169,2,1,11/19/2024,41.76,7.7,Credit Card,None,None
9,978,2,103,4917,1324,1,1/27/2025,135.63,3.3,Credit Card,None,None


In [10]:
sales_data['Discount Reason'].value_counts(dropna=False)

Discount Reason
None                      8268
Promo code                 841
Competitor Price Match     456
Loyalty Discount           335
Name: count, dtype: int64

##### 2) Customer Master Table

In [11]:
customer_data.head(10)

,Customer Id,Full Name,Email,Phone,City,Signup Date,Notes
0,943,Jason Nguyen,jasonnguyen943@hotmail.com,(177) 788-9646,TORONTO,3/26/2021,NaN
1,645,Michelle Chan,michelle.chan645@gmail.com,9612734670,TORONTO,4/25/2022,NaN
2,1004,Jennifer Smith,jennifer_smith1004@outlook.com,425-381-0689,TORONTO,8/23/2022,NaN
3,249,James Singh,jamessingh249@yahoo.com,(517) 845-3144,TORONTO,9/27/2022,NaN
4,1046,Michael Wilson,michael_wilson1046@yahoo.com,3511582388,TORONTO,4/11/2023,NaN
5,1532,Susan Cllark,susan_cllark1532@gmail.com,(251) 983-6473,TORONTO,12/6/2022,NaN
6,629,Robert White,robert.white629@yahoo.com,(886) 784-1047,TORONTO,7/30/2024,NaN
7,992,Mary Gagnon,marygagnon992@outlook.com,944.547.3221,TORONTO,10/22/2020,NaN
8,322,Joseph Taylor,joseph.taylor322@yahoo.com,981.019.4771,TORONTO,3/14/2021,NaN
9,182,Jessica Clark,jessica_clark182@outlook.com,1918453122,TORONTO,5/4/2019,NaN


In [12]:
customer_data['Notes'].unique()

array([nan, 'Prefers text reminders', 'Opted out of email marketing',
       'Gold member - loyalty since 2024',
       'Gold member - loyalty since 2020',
       'Platinum member - loyalty since 2022',
       'Gold member - loyalty since 2021',
       'Gold member - loyalty since 2023', 'Requested large-print labels',
       'Prefers pickup, not delivery',
       'Platinum member - loyalty since 2021',
       'Silver member - loyalty since 2022',
       'Silver member - loyalty since 2019',
       'Gold member - loyalty since 2025',
       'Gold member - loyalty since 2019',
       'Platinum member - loyalty since 2023',
       'Silver member - loyalty since 2021',
       'Silver member - loyalty since 2024',
       'Platinum member - loyalty since 2024',
       'Platinum member - loyalty since 2019',
       'Silver member - loyalty since 2023',
       'Silver member - loyalty since 2020',
       'Platinum member - loyalty since 2020',
       'Platinum member - loyalty since 2025',
  

In [13]:
def extract_loyalty(note):
    if pd.isnull(note) or note ==" ":
        return pd.Series({'Loyalty Tier':None , 'Loyalty Since':None})
    loyalty_tier_match=re.search(r'(Silver|Gold|Platinum) member - loyalty since (\d{4})',note)
    if loyalty_tier_match:
        return pd.Series({'Loyalty Tier':loyalty_tier_match.group(1), 'Loyalty Since':int(loyalty_tier_match.group(2))})
    return pd.Series({'Loyalty Tier':None , 'Loyalty Since':None})

extracted_loyalty=customer_data['Notes'].apply(extract_loyalty)
customer_data=pd.concat([customer_data,extracted_loyalty],axis=1)


In [14]:
customer_data.head(10)

,Customer Id,Full Name,Email,Phone,City,Signup Date,Notes,Loyalty Tier,Loyalty Since
0,943,Jason Nguyen,jasonnguyen943@hotmail.com,(177) 788-9646,TORONTO,3/26/2021,NaN,None,NaN
1,645,Michelle Chan,michelle.chan645@gmail.com,9612734670,TORONTO,4/25/2022,NaN,None,NaN
2,1004,Jennifer Smith,jennifer_smith1004@outlook.com,425-381-0689,TORONTO,8/23/2022,NaN,None,NaN
3,249,James Singh,jamessingh249@yahoo.com,(517) 845-3144,TORONTO,9/27/2022,NaN,None,NaN
4,1046,Michael Wilson,michael_wilson1046@yahoo.com,3511582388,TORONTO,4/11/2023,NaN,None,NaN
5,1532,Susan Cllark,susan_cllark1532@gmail.com,(251) 983-6473,TORONTO,12/6/2022,NaN,None,NaN
6,629,Robert White,robert.white629@yahoo.com,(886) 784-1047,TORONTO,7/30/2024,NaN,None,NaN
7,992,Mary Gagnon,marygagnon992@outlook.com,944.547.3221,TORONTO,10/22/2020,NaN,None,NaN
8,322,Joseph Taylor,joseph.taylor322@yahoo.com,981.019.4771,TORONTO,3/14/2021,NaN,None,NaN
9,182,Jessica Clark,jessica_clark182@outlook.com,1918453122,TORONTO,5/4/2019,NaN,None,NaN


In [15]:
customer_data['Loyalty Tier'].value_counts(dropna=False)

Loyalty Tier
None        1099
Silver       158
Platinum     154
Gold         144
Name: count, dtype: int64

In [16]:
customer_data.drop(columns=['Notes'],inplace=True)

In [17]:
customer_data['City'].unique()

array(['TORONTO', 'Vancouver', 'Calgary', 'Montreal', 'Winnipeg, MB',
       'waterloo', 'Quebec City', 'Winnipeg', 'Vancouver, BC', 'Edmonton',
       'Halifax', 'Moncton', 'Saskatoon', 'Ottawa, ON', 'Montreal, QC',
       'London', 'Regina', 'Ottawa', 'Kitchener', 'Hamilton',
       'Calgary, AB', 'Quebec', 'Edmonton, AB', 'Vancouvr', 'QC City',
       'Regina, SK', 'Montréal', 'Halifax, NS', 'Waterloo, ON',
       'Toronto, ON', 'Trnto', 'Moncton, NB'], dtype=object)

In [18]:
# Cleaning the 'City' column by removing leading/trailing spaces and duplicates formats 
customer_data['City Clean']=customer_data['City'].astype(str).str.split(',').str[0].str.strip().str.title()

In [19]:
customer_data['City Clean'].unique()

array(['Toronto', 'Vancouver', 'Calgary', 'Montreal', 'Winnipeg',
       'Waterloo', 'Quebec City', 'Edmonton', 'Halifax', 'Moncton',
       'Saskatoon', 'Ottawa', 'London', 'Regina', 'Kitchener', 'Hamilton',
       'Quebec', 'Vancouvr', 'Qc City', 'Montréal', 'Trnto'], dtype=object)

In [20]:
customer_data['Phone'].unique()

array(['(177) 788-9646', '9612734670', '425-381-0689', ..., '5896937041',
       '3283459480', '1167412093'], dtype=object)

In [21]:
# Cleaning the 'Phone' column by removing non-digit characters
customer_data['Phone Clean']=customer_data['Phone'].astype(str).str.replace(r'\D','',regex=True)

In [22]:
customer_data['Phone Clean']

0       1777889646
1       9612734670
2       4253810689
3       5178453144
4       3511582388
           ...    
1550    8904907151
1551    1925853016
1552    5896937041
1553    3283459480
1554    1167412093
Name: Phone Clean, Length: 1555, dtype: object

In [23]:
customer_data['Full Name'].unique()

array(['Jason Nguyen', 'Michelle Chan', 'Jennifer Smith', 'James Singh',
       'Michael Wilson', 'Susan Cllark', 'Robert White', 'Mary Gagnon',
       'Joseph Taylor', 'Jessica Clark', 'Emily Lee', 'Emily Young',
       'Daniel White', 'John Roy', 'David Martin', 'Jessica Singh',
       'Sarah Clark', 'Brian Campbell', 'Linda Hall', 'Jason Young',
       'Sarah Tremblay', 'Sarah MacDonald', 'Joseph Campbell',
       'Laura Taylor', 'Laura Hall', 'Matthew Smith', 'Andrew White',
       'Lisa Brown', 'Susan Chan', 'Lisa Martin', 'Laura Patel',
       'Jennifer Gagnon', 'Amanda Chan', 'Linda Taylor', 'Robert Gagnon',
       'Elizabeth Nguyen', 'Patricia Hall', 'Susan MacDonald',
       'Brian Brown', 'Justin Hall', 'Sarah Brown', 'John Clark',
       'Karen Tremblay', 'Amanda Anderson', 'Kevin Campbell',
       'Emily Taylor', 'Emily Tremblay', 'Karen Campbell', 'Laura White',
       'John Brown', 'Jessica Brown', 'Jennifer Wilson', 'Susan Wilson',
       'Daniel Singh', 'Joseph Singh', 

In [24]:
# Check the number of duplicate phone numbers in the 'Phone Clean' column
duplicates_groups=[
    {
        'phone': phone,
        'customer_ids':group['Customer Id'].tolist()
    }

    for phone,group in customer_data.groupby('Phone Clean')
    if len(group)>1
]

duplicates_groups

[{'phone': '0120312738', 'customer_ids': [1529, 1368]},
 {'phone': '0346718657', 'customer_ids': [1523, 1262]},
 {'phone': '0903958146', 'customer_ids': [227, 1539]},
 {'phone': '0940211340', 'customer_ids': [1544, 843]},
 {'phone': '1070983552', 'customer_ids': [1288, 1551]},
 {'phone': '1232566042', 'customer_ids': [11, 1507]},
 {'phone': '1387506072', 'customer_ids': [1511, 490]},
 {'phone': '1403780254', 'customer_ids': [201, 1545]},
 {'phone': '1918453122', 'customer_ids': [182, 1502]},
 {'phone': '2082302039', 'customer_ids': [92, 1542]},
 {'phone': '2364021542', 'customer_ids': [1505, 1308]},
 {'phone': '2519836473', 'customer_ids': [1532, 43]},
 {'phone': '2719200342', 'customer_ids': [1536, 1363]},
 {'phone': '2861539636', 'customer_ids': [1543, 872]},
 {'phone': '2930495031', 'customer_ids': [1541, 1492]},
 {'phone': '3070107286', 'customer_ids': [1518, 1453]},
 {'phone': '3100486159', 'customer_ids': [85, 1517]},
 {'phone': '3122820605', 'customer_ids': [1525, 426]},
 {'phon

In [25]:
# # Dropping duplicate rows based on the 'Phone Clean' column
# customer_data=customer_data.drop_duplicates(subset='Phone Clean',keep='first')

##### 3) Employees Table

In [26]:
employees_data.head(10)

,Employee Id,Store Id,Employee Name,Email,Position,Salary,Hire Date,Is Pharmacist
0,1,4,Staff Member 1,staff_member_1@company.com,Sales Associate,8830.50,5/6/2020,0
1,2,9,Staff Member 2,staff_member_2@company.com,Inventory Specialist,6570.51,5/27/2024,0
2,3,1,Staff Member 3,staff_member_3@company.com,Inventory Specialist,6690.47,12/3/2023,0
3,4,1,Staff Member 4,staff_member_4@company.com,Sales Associate,4809.30,10/13/2019,1
4,5,7,Staff Member 5,staff_member_5@company.com,Sales Associate,8930.54,1/4/2018,0
5,6,9,Staff Member 6,staff_member_6@company.com,Inventory Specialist,5889.70,4/29/2020,0
6,7,7,Staff Member 7,staff_member_7@company.com,Inventory Specialist,5187.55,2/8/2018,0
7,8,2,Staff Member 8,staff_member_8@company.com,Store Manager,7443.98,10/22/2023,0
8,9,8,Staff Member 9,staff_member_9@company.com,Inventory Specialist,9289.79,8/22/2024,1
9,10,2,Staff Member 10,staff_member_10@company.com,Pharmacist,6057.50,11/11/2018,1


In [27]:
employees_data.loc[
    employees_data['Is Pharmacist'] == 0,
    ['Position', 'Is Pharmacist']
]

,Position,Is Pharmacist
0,Sales Associate,0
1,Inventory Specialist,0
2,Inventory Specialist,0
4,Sales Associate,0
5,Inventory Specialist,0
...,...,...
114,Technician,0
115,Sales Associate,0
116,Pharmacist,0
118,Store Manager,0


In [28]:
employees_data['Position'].unique()

array(['Sales Associate', 'Inventory Specialist', 'Store Manager',
       'Pharmacist', 'Technician'], dtype=object)

In [29]:
employees_data['Is Pharmacist'] = (
    employees_data['Position'] == 'Pharmacist'
).astype(int)

In [30]:
employees_data['Is Pharmacist'].value_counts()

Is Pharmacist
0    100
1     20
Name: count, dtype: int64

##### 4) Product Catalog Table

In [31]:
products_data.head(10)

,Product Id,Product Name,Category Id,Supplier Id,Unit Price,Cost Price,Brand,Expiry Date,Storage Condition,Created At,Pack Info
0,2,Health Product 2,8,3,111.42,64.75,MediCare,8/8/2026,Refrigerated,9/13/2022,NaN
1,39,Health Product 39,9,5,34.55,21.04,BioZen,12/29/2025,Refrigerated,8/6/2025,NaN
2,51,Health Product 51,3,2,146.82,89.56,CuraMax,9/17/2026,Refrigerated,9/7/2025,NaN
3,76,Health Product 76,1,3,154.35,60.18,LifeLine,8/23/2026,Refrigerated,9/3/2024,NaN
4,123,Health Product 123,8,4,299.65,174.20,CuraMax,11/12/2028,Refrigerated,10/31/2025,NaN
5,136,Health Product 136,4,1,208.37,138.08,PharmaCo,10/8/2026,Refrigerated,11/30/2021,NaN
6,255,Health Product 255,1,2,94.36,34.91,LifeLine,11/17/2028,Refrigerated,12/20/2022,NaN
7,303,Health Product 303,6,4,262.69,133.45,CuraMax,5/20/2026,Refrigerated,10/28/2023,NaN
8,352,Health Product 352,7,4,16.99,8.02,HealthPlus,12/8/2029,Refrigerated,4/10/2025,NaN
9,360,Health Product 360,2,4,37.57,15.69,HealthPlus,11/4/2026,Refrigerated,8/23/2024,NaN


In [32]:
products_data['Pack Info'].unique()

array([nan, 'Pack: 300ml | Strength: 250mg|5ml | Form: Liquid',
       'Pack: 90g | Form: Cream',
       'Pack: 20 tabs | Strength: 250mg | Form: Tablet',
       'Pack: 60g | Form: Cream',
       'Pack: 20 tabs | Strength: 50mg | Form: Tablet',
       'Pack: 20 units | Form: Kit', 'Pack: 100 units | Form: Kit',
       'Pack: 90 tabs | Strength: 50mg | Form: Tablet',
       'Form: Liquid | Pack: 150ml | Strength: 50mg|5ml',
       'Pack: 30 caps | Strength: 25mg | Form: Capsule',
       'Pack: 30 tabs | Strength: 650mg | Form: Tablet',
       'Pack: 30 tabs | Strength: 250mg | Form: Tablet',
       'Pack: 30 tabs | Strength: 25mg | Form: Tablet',
       'Pack: 100 tabs | Strength: 250mg | Form: Tablet', 'Form: Device',
       'Pack: 450ml | Strength: 25mg|5ml | Form: Liquid',
       'Pack: 30g | Form: Cream',
       'Pack: 60 tabs | Strength: 100mg | Form: Tablet', 'Pack: 10g',
       'Pack: 20g | Form: Cream', 'Pack: 90g | Form: Bandage',
       'Pack: 30 caps | Strength: 500mg | Form:

In [33]:
def extract_pack_info(pack_info):
    
    if pd.isnull(pack_info) or str(pack_info).strip() == "":
        return pd.Series({'Pack Size': None, 'Strength': None, 'Form': None})
    
    pack_info_str = str(pack_info)

    pack_size_match = re.search(r'Pack:\s*([^|]+)', pack_info_str)
    strength_match = re.search(r'Strength:\s*([^|]+)', pack_info_str)
    form_match = re.search(r'Form:\s*([^|]+)', pack_info_str)

    return pd.Series({
        'Pack Size': pack_size_match.group(1).strip() if pack_size_match else None,
        'Strength': strength_match.group(1).strip() if strength_match else None,
        'Form': form_match.group(1).strip() if form_match else None
    })


extracted_pack_info = products_data['Pack Info'].apply(extract_pack_info)
products_data = pd.concat([products_data, extracted_pack_info], axis=1)

In [34]:
products_data['Form'].value_counts(dropna=False)

Form
None       913
Tablet     896
Cream      888
Kit        502
Device     500
Capsule    447
Bandage    428
Liquid     426
Name: count, dtype: int64

In [35]:
products_data.drop(columns=['Pack Info'], inplace=True)

In [36]:
products_data = products_data.merge(
    category_data[['Category Id', 'Category Name']], 
    on='Category Id', 
    how='left'
)

In [37]:
products_data

,Product Id,Product Name,Category Id,Supplier Id,Unit Price,Cost Price,Brand,Expiry Date,Storage Condition,Created At,Pack Size,Strength,Form,Category Name
0,2,Health Product 2,8,3,111.42,64.75,MediCare,8/8/2026,Refrigerated,9/13/2022,None,None,None,Skin & Derma Care
1,39,Health Product 39,9,5,34.55,21.04,BioZen,12/29/2025,Refrigerated,8/6/2025,None,None,None,Diagnostics
2,51,Health Product 51,3,2,146.82,89.56,CuraMax,9/17/2026,Refrigerated,9/7/2025,None,None,None,Vitamins & Supplements
3,76,Health Product 76,1,3,154.35,60.18,LifeLine,8/23/2026,Refrigerated,9/3/2024,None,None,None,Prescription Drugs
4,123,Health Product 123,8,4,299.65,174.20,CuraMax,11/12/2028,Refrigerated,10/31/2025,None,None,None,Skin & Derma Care
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4944,Health Product 4944,2,2,57.99,23.14,BioZen,9/8/2029,Ambient,8/1/2025,100 tabs,100mg,Tablet,OTC Pain Relief
4996,4946,Health Product 4946,1,2,294.70,176.63,CuraMax,11/27/2029,Ambient,8/8/2023,90 tabs,250mg,Tablet,Prescription Drugs
4997,4959,Health Product 4959,4,2,279.36,175.88,HealthPlus,11/2/2029,Ambient,10/25/2025,20g,None,Cream,Personal Care
4998,4963,Health Product 4963,7,2,83.39,48.87,HealthPlus,2/24/2029,Ambient,1/14/2025,10g,None,Bandage,First Aid


##### 5) Stock Condition Table

In [38]:
stock_data.head(10)

,Stock Id,Store Id,Product Id,Quantity,Reorder Level,last Restocked,Batch Number,Expiry Date,Received Date,Condition Notes
0,37,10,2214,28,31,2/23/2026,BATCH-37,1/9/2030,2/23/2026,NaN
1,61,10,755,114,48,9/11/2025,BATCH-61,2/6/2029,9/11/2025,NaN
2,109,10,4892,59,24,12/3/2025,BATCH-109,8/10/2028,12/3/2025,NaN
3,132,10,373,10,35,8/25/2025,BATCH-132,8/1/2028,8/25/2025,NaN
4,137,10,33,90,15,8/15/2026,BATCH-137,11/1/2027,8/15/2026,NaN
5,187,10,554,27,20,6/18/2026,BATCH-187,3/12/2028,6/18/2026,NaN
6,194,10,4389,93,41,3/23/2026,BATCH-194,5/15/2028,3/23/2026,NaN
7,205,10,4531,98,17,1/4/2026,BATCH-205,3/26/2028,1/4/2026,NaN
8,212,10,4379,42,33,11/27/2025,BATCH-212,11/28/2026,11/27/2025,NaN
9,269,10,3684,103,59,3/22/2026,BATCH-269,3/10/2029,3/22/2026,NaN


In [39]:
stock_data['Condition Notes'].unique()

array([nan, 'Expiring in 0 days - flag for clearance',
       'Expiring in 11 days - flag for clearance',
       'Expiring in 17 days - flag for clearance',
       'Below reorder level - reorder placed 2025-11-10',
       'Below reorder level - reorder placed 2025-10-30',
       'Below reorder level - reorder placed 2026-04-11',
       'Expiring in 0 days - flag for clearance; Below reorder level - reorder placed 2025-12-08',
       'Below reorder level - reorder placed 2025-12-24',
       'Below reorder level - reorder placed 2026-06-22',
       'Expiring in 45 days - monitor; Below reorder level - reorder placed 2026-04-27',
       'Below reorder level - reorder placed 2026-09-03',
       'Expiring in 0 days - flag for clearance; Below reorder level - reorder placed 2025-07-21',
       'Below reorder level - reorder placed 2025-10-05',
       'Below reorder level - reorder placed 2026-07-08',
       'Below reorder level - reorder placed 2026-07-27',
       'Below reorder level - reor

In [40]:
# Extracting information from the 'Condition Notes' column
def extract_condition(note):
    if pd.isnull(note) or note.strip() == "":
        return pd.Series({'Expiry Risk Flag':False, 'Days To Expiry Flagged':None,'Below Recorder Flag':False,'Recorder Placed Date':None})

    expiry_match=re.search(r'Expiring in (\d+) days', note)
    recorder_match=re.search(r'Below reorder level - reorder placed (\d{4}-\d{2}-\d{2})', note)  

    return pd.Series({'Expiry Risk Flag':bool(expiry_match),
                      'Days To Expiry Flagged':int(expiry_match.group(1)) if expiry_match else None,
                      'Below Recorder Flag':bool(recorder_match),
                      'Recorder Placed Date':recorder_match.group(1) if recorder_match else None})

extracted_condition=stock_data['Condition Notes'].apply(extract_condition)
stock_data=pd.concat([stock_data,extracted_condition],axis=1)


In [41]:
stock_data.drop(columns=['Condition Notes'],inplace=True)

In [42]:
stock_data.head(10)

,Stock Id,Store Id,Product Id,Quantity,Reorder Level,last Restocked,Batch Number,Expiry Date,Received Date,Expiry Risk Flag,Days To Expiry Flagged,Below Recorder Flag,Recorder Placed Date
0,37,10,2214,28,31,2/23/2026,BATCH-37,1/9/2030,2/23/2026,False,NaN,False,None
1,61,10,755,114,48,9/11/2025,BATCH-61,2/6/2029,9/11/2025,False,NaN,False,None
2,109,10,4892,59,24,12/3/2025,BATCH-109,8/10/2028,12/3/2025,False,NaN,False,None
3,132,10,373,10,35,8/25/2025,BATCH-132,8/1/2028,8/25/2025,False,NaN,False,None
4,137,10,33,90,15,8/15/2026,BATCH-137,11/1/2027,8/15/2026,False,NaN,False,None
5,187,10,554,27,20,6/18/2026,BATCH-187,3/12/2028,6/18/2026,False,NaN,False,None
6,194,10,4389,93,41,3/23/2026,BATCH-194,5/15/2028,3/23/2026,False,NaN,False,None
7,205,10,4531,98,17,1/4/2026,BATCH-205,3/26/2028,1/4/2026,False,NaN,False,None
8,212,10,4379,42,33,11/27/2025,BATCH-212,11/28/2026,11/27/2025,False,NaN,False,None
9,269,10,3684,103,59,3/22/2026,BATCH-269,3/10/2029,3/22/2026,False,NaN,False,None


In [43]:
sales_data.columns

Index(['Sale Id', 'Store Id', 'Employee Id', 'Product Id', 'Customer Id',
       'Quantity', 'Sale Date', 'Unit Price', 'Discount Pct', 'Payment Method',
       'Promo Code', 'Discount Reason'],
      dtype='object')

#### Feature Engineering

In [44]:
sales_data['Total Amount']=(sales_data['Quantity'] * sales_data['Unit Price'] * (1 - sales_data['Discount Pct']/100)).round(2)

In [45]:
sales_data=sales_data.merge(products_data[['Product Id', 'Cost Price']])

In [46]:
sales_data['Cost Amount']=(sales_data['Quantity'] * sales_data['Cost Price']).round(2)

In [47]:
sales_data['Profit']=(sales_data['Total Amount'] - sales_data['Cost Amount']).round(2)

In [48]:
sales_data['Profit Margin Pct']=(sales_data['Profit']/sales_data['Total Amount']*100).round(2)

#### Data Validation

###### 1) Null Validation

In [49]:
def null_validation(data,table_name):
    nulls=data.isnull().sum()
    nulls=nulls[nulls>0]

    if len(nulls)==0:
        print(f'{table_name} : No Null values')
    else:
        print(f'{table_name} : Null values found')    

In [50]:
null_validation(customer_data,'CustomerMaster')
null_validation(sales_data,'AllSales')
null_validation(category_data,'Categories')
null_validation(employees_data,'Employees')
null_validation(prescriptions_data,'Prescriptions')
null_validation(products_data,'ProductCatalog')
null_validation(stock_data,'Stock')
null_validation(stores_data,'Stores')
null_validation(suppliers_data,'Suppliers')

CustomerMaster : Null values found
AllSales : Null values found
Categories : No Null values
Employees : No Null values
Prescriptions : No Null values
ProductCatalog : Null values found
Stock : Null values found
Stores : No Null values
Suppliers : No Null values


###### 2) Duplicates Validation

In [51]:
def duplicates_validation(data,table_name):
    duplicates=data.duplicated().sum()

    if duplicates==0:
        print(f'{table_name} : No duplicates')
    else:
        print(f'{table_name} : Duplicated found')    

In [52]:
duplicates_validation(customer_data,'CustomerMaster')
duplicates_validation(sales_data,'AllSales')
duplicates_validation(category_data,'Categories')
duplicates_validation(employees_data,'Employees')
duplicates_validation(prescriptions_data,'Prescriptions')
duplicates_validation(products_data,'ProductCatalog')
duplicates_validation(stock_data,'Stock')
duplicates_validation(stores_data,'Stores')
duplicates_validation(suppliers_data,'Suppliers')

CustomerMaster : No duplicates
AllSales : No duplicates
Categories : No duplicates
Employees : No duplicates
Prescriptions : No duplicates
ProductCatalog : No duplicates
Stock : No duplicates
Stores : No duplicates
Suppliers : No duplicates


###### 3) Data Types Validation

In [53]:
def data_types_validation(data,table_name):
    print(f'\n-----{table_name}-----')
    print(data.dtypes)

In [54]:
data_types_validation(customer_data,'CustomerMaster')
data_types_validation(sales_data,'AllSales')
data_types_validation(category_data,'Categories')
data_types_validation(employees_data,'Employees')
data_types_validation(prescriptions_data,'Prescriptions')
data_types_validation(products_data,'ProductCatalog')
data_types_validation(stock_data,'Stock')
data_types_validation(stores_data,'Stores')
data_types_validation(suppliers_data,'Suppliers')


-----CustomerMaster-----
Customer Id        int64
Full Name         object
Email             object
Phone             object
City              object
Signup Date       object
Loyalty Tier      object
Loyalty Since    float64
City Clean        object
Phone Clean       object
dtype: object

-----AllSales-----
Sale Id                int64
Store Id               int64
Employee Id            int64
Product Id             int64
Customer Id            int64
Quantity               int64
Sale Date             object
Unit Price           float64
Discount Pct         float64
Payment Method        object
Promo Code            object
Discount Reason       object
Total Amount         float64
Cost Price           float64
Cost Amount          float64
Profit               float64
Profit Margin Pct    float64
dtype: object

-----Categories-----
Category Id       int64
Category Name    object
dtype: object

-----Employees-----
Employee Id        int64
Store Id           int64
Employee Name     object
Ema

###### 4) Referential Integrity Validation

In [55]:
def check_fk(child_data,child_col,parent_data,parent_col):
    invalid=child_data[~child_data[child_col].isin(parent_data[parent_col])]

    if invalid.empty:
        print('Referential Integrity valid')
    else:
        print(f'len{invalid} invalid values found')    

In [56]:
check_fk(sales_data,'Store Id',stores_data,'Store Id')
check_fk(sales_data,'Employee Id',employees_data,'Employee Id')
check_fk(sales_data,'Product Id',products_data,'Product Id')
check_fk(sales_data,'Customer Id',customer_data,'Customer Id')

Referential Integrity valid
Referential Integrity valid
Referential Integrity valid
Referential Integrity valid


In [57]:
# Saving All Tables After Cleaning
sales_data.to_csv("F:\A7med els3eed\Digitara - Data Analysis\Health Retail Stores\Cleaned Data\AllSales_Cleaned.csv", index=False)
customer_data.to_csv("F:\A7med els3eed\Digitara - Data Analysis\Health Retail Stores\Cleaned Data\CustomerMaster_Cleaned.csv",index=False)
employees_data.to_csv("F:\A7med els3eed\Digitara - Data Analysis\Health Retail Stores\Cleaned Data\Employees_Cleaned.csv",index=False) 
products_data.to_csv("F:\A7med els3eed\Digitara - Data Analysis\Health Retail Stores\Cleaned Data\ProductCatalog_Cleaned.csv", index=False)
stock_data.to_csv("F:\A7med els3eed\Digitara - Data Analysis\Health Retail Stores\Cleaned Data\Stock_Cleaned.csv", index=False)
prescriptions_data.to_csv("F:\A7med els3eed\Digitara - Data Analysis\Health Retail Stores\Cleaned Data\Prescriptions_Cleaned.csv",index=False)
category_data.to_csv("F:\A7med els3eed\Digitara - Data Analysis\Health Retail Stores\Cleaned Data\Categories_Cleaned.csv",index=False)
stores_data.to_csv("F:\A7med els3eed\Digitara - Data Analysis\Health Retail Stores\Cleaned Data\Stores_Cleaned.csv",index=False)
suppliers_data.to_csv("F:\A7med els3eed\Digitara - Data Analysis\Health Retail Stores\Cleaned Data\Suppliers_Cleaned.csv",index=False)


I did the rest of the cleaning using the power Query